In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
DATASET_NAME = "alstdi"  # options: "alstdi", "parkceleb"

metadata_path = Path(f"../data/metadata/{DATASET_NAME}_metadata.csv")
split_path = Path(f"../data/splits/{DATASET_NAME}_split.json")

metadata_path, split_path

In [ ]:
df = pd.read_csv(metadata_path)

print(f"Number of recordings: {len(df)}")
print(f"Number of subjects: {df['subject_id'].nunique()}")

df.head()

In [ ]:
required_columns = [
    "subject_id",
    "audio_path",
    "transcript",
    "timestamp",
    "label",
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("All required columns are present.")

In [ ]:
sessions_per_subject = df.groupby("subject_id").size()

summary = {
    "num_subjects": df["subject_id"].nunique(),
    "num_recordings": len(df),
    "avg_recordings_per_subject": sessions_per_subject.mean(),
    "min_recordings_per_subject": sessions_per_subject.min(),
    "max_recordings_per_subject": sessions_per_subject.max(),
    "label_min": df["label"].min(),
    "label_max": df["label"].max(),
    "label_mean": df["label"].mean(),
}

pd.DataFrame([summary])

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["label"], bins=20)
plt.xlabel("Normalized progression label")
plt.ylabel("Number of recordings")
plt.title("Progression Label Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(sessions_per_subject, bins=20)
plt.xlabel("Number of recordings per subject")
plt.ylabel("Number of subjects")
plt.title("Longitudinal Recording Counts")
plt.show()

In [ ]:
with open(split_path, "r") as f:
    split = json.load(f)

train_subjects = set(split["train"])
test_subjects = set(split["test"])

print(f"Train subjects: {len(train_subjects)}")
print(f"Test subjects: {len(test_subjects)}")
print(f"Overlap: {len(train_subjects & test_subjects)}")

In [ ]:
assert len(train_subjects & test_subjects) == 0, "Train/test subject leakage detected."

metadata_subjects = set(df["subject_id"].astype(str))
missing_from_metadata = (train_subjects | test_subjects) - metadata_subjects

print(f"Subjects in split but missing from metadata: {len(missing_from_metadata)}")

In [ ]:
example_subjects = (
    df.groupby("subject_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .index
)

plt.figure(figsize=(8, 4))

 for_idx = 0
for subject_id in example_subjects:
    sub = df[df["subject_id"] == subject_id].sort_values("timestamp")
    plt.plot(sub["timestamp"], [for_idx] * len(sub), marker="o", linestyle="-", label=str(subject_id))
    for_idx += 1

plt.xlabel("Timestamp")
plt.ylabel("Example subject")
plt.title("Example Longitudinal Timelines")
plt.yticks([])
plt.legend()
plt.show()